In [11]:

import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.utils import pad_sequences
from tensorflow.keras.layers import Input, Embedding, GRU, Dense
from tensorflow.keras.models import Model

tf.random.set_seed(42)
print("TF:", tf.__version__, "| GPUs:", tf.config.list_physical_devices("GPU"))

TF: 2.20.0 | GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]


In [12]:
## now add the path of the english and urdu file 
ENG_file="/kaggle/input/datasets/muhammadarshadkarbog/english-to-ur-translation/english-corpus.txt"
UR_file="/kaggle/input/datasets/muhammadarshadkarbog/english-to-ur-translation/urdu-corpus.txt"


In [13]:
### now we want to zip this two file 
en_file=open(ENG_file,encoding=('utf-8').strip()).read().split('\n')
ur_file=open(UR_file,encoding=('utf-8').strip()).read().split('\n')
### now make the pair that english Sentance correspond with urdu sentance
pairs=[(e.strip(),u.strip()) for e ,u in zip(en_file,ur_file) if e.strip() and u.strip()]
print("Total pair is",len(pairs))
for e,u in pairs[:5]:
    print(f"{e:30}->{u}")


Total pair is 24524
is zain your nephew           ->زین تمہارا بھتیجا ہے۔
i wish youd trust me          ->کاش تم مجھ پر بھروسہ کرتے
did he touch you              ->کیا اس نے آپ کو چھوا؟
its part of life              ->اس کی زندگی کا حصہ
zain isnt ugly                ->زین بدصورت نہیں ہے۔


In [14]:
# pairs=[]
# for e,u in zip(en_file,ur_file):
#     pair=(e.strip(),u.strip())
#     if u.strip() and e.strip():
#         pairs.append(pair)

# print("The total pair is ",len(pairs))
# for e,u in pairs[:5]:
#     print(f"{e:30s}-->{u}")


In [15]:
print(type(ENG_file))
print(type(UR_file))

<class 'str'>
<class 'str'>


In [16]:
# ## we take the maximum pair is 8000 for faster traning
# max_pair=8000
# pair= [(e, u) for e, u in pairs if len(e.split()) <= 6 and len(u.split()) <= 8][:max_pair]
# en_texts = [e for e, u in pair]

# print("using", len(pair), "pairs")
# print("example target:", ur_texts[0])


In [17]:
max_pair = 8000

pair = []

for e, u in pairs:
    if len(e.split()) <= 6 and len(u.split()) <= 8:
        pair.append((e, u))

pair = pair[:max_pair]

print("Total pairs:", len(pair))
print(pair[:5])

Total pairs: 8000
[('is zain your nephew', 'زین تمہارا بھتیجا ہے۔'), ('i wish youd trust me', 'کاش تم مجھ پر بھروسہ کرتے'), ('did he touch you', 'کیا اس نے آپ کو چھوا؟'), ('its part of life', 'اس کی زندگی کا حصہ'), ('zain isnt ugly', 'زین بدصورت نہیں ہے۔')]


In [18]:
en_tok = Tokenizer()
en_tok.fit_on_texts(en_file)
en_vocab = len(en_tok.word_index) + 1

In [19]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# ==========================================
# Urdu Text
# ==========================================

ur_texts = [
    "<start> " + u + " <end>"
    for e, u in pair
]

# IMPORTANT:
# < and > ko remove nahi karna
ur_tok = Tokenizer(
    filters='!"#$%&()*+,-./:;=?@[\\]^_`{|}~\t\n'
)

ur_tok.fit_on_texts(ur_texts)

# ==========================================
# Urdu sequences
# ==========================================

ur_seq = ur_tok.texts_to_sequences(ur_texts)

# ==========================================
# Vocabulary
# ==========================================

ur_vocab = len(ur_tok.word_index) + 1

# ==========================================
# Check
# ==========================================

start_id = ur_tok.word_index["<start>"]
end_id = ur_tok.word_index["<end>"]

print("START ID:", start_id)
print("END ID:", end_id)

print("\nFirst sentence:")
print(ur_texts[0])

print("\nFirst sequence:")
print(ur_seq[0])

print("\nVocabulary:", ur_vocab)

START ID: 1
END ID: 2

First sentence:
<start> زین تمہارا بھتیجا ہے۔ <end>

First sequence:
[1, 5, 384, 1295, 18, 2]

Vocabulary: 3442


In [21]:
en_seq = en_tok.texts_to_sequences(en_file)
# ur_seq = en_tok.texts_to_sequences(ur_texts)
ur_seq = ur_tok.texts_to_sequences(ur_texts)

print("New sequence:")
print(ur_seq[0])

New sequence:
[1, 5, 384, 1295, 18, 2]


In [22]:
MAX_EN = max(len(s) for s in en_seq)
MAX_UR = max(len(s) for s in ur_seq)
en_seq = pad_sequences(en_seq, maxlen=MAX_EN, padding="post")
ur_seq = pad_sequences(ur_seq, maxlen=MAX_UR, padding="post")
print("English padded:", en_seq.shape, "| Urdu padded:", ur_seq.shape)
print("Padded:")
print(ur_seq[0])


English padded: (24526, 14) | Urdu padded: (8000, 10)
Padded:
[   1    5  384 1295   18    2    0    0    0    0]


In [23]:
decoder_input_data = ur_seq[:, :-1]
decoder_target = ur_seq[:, 1:]

print("Decoder input:")
print(decoder_input_data[0])

print("Decoder target:")
print(decoder_target[0])

Decoder input:
[   1    5  384 1295   18    2    0    0    0]
Decoder target:
[   5  384 1295   18    2    0    0    0    0]


In [24]:
MAX_EN

14

In [25]:
# ==========================================
# Urdu padding
# ==========================================

ur_seq_pad = pad_sequences(
    ur_seq,
    maxlen=MAX_UR,
    padding="post"
)

print("Padded Urdu shape:", ur_seq_pad.shape)
print("First padded sequence:", ur_seq_pad[0])


# ==========================================
# Decoder input and target
# ==========================================

decoder_input = ur_seq_pad[:, :-1]
decoder_target = ur_seq_pad[:, 1:]

print("Decoder input :", decoder_input.shape)
print("Decoder target:", decoder_target.shape)
print("Sentence:")
print(ur_texts[0])

print("\nCurrent tokenizer:")
print(ur_tok.texts_to_sequences([ur_texts[0]]))

print("\nCurrent word IDs:")
for word in ur_texts[0].split():
    print(repr(word), "=>", ur_tok.word_index.get(word))

Padded Urdu shape: (8000, 10)
First padded sequence: [   1    5  384 1295   18    2    0    0    0    0]
Decoder input : (8000, 9)
Decoder target: (8000, 9)
Sentence:
<start> زین تمہارا بھتیجا ہے۔ <end>

Current tokenizer:
[[1, 5, 384, 1295, 18, 2]]

Current word IDs:
'<start>' => 1
'زین' => 5
'تمہارا' => 384
'بھتیجا' => 1295
'ہے۔' => 18
'<end>' => 2


In [26]:
LATENT=256
EMBADING_DIM=100
### ENCODER archectiture
enc_input=Input(shape=(MAX_EN,),name='encoder gru input')
### is input layer main hum MAx_eng ka senatnce dengi
### now we will  make the Encoder embading layer 
enc_emb=Embedding(en_vocab,EMBADING_DIM)(enc_input)
### is emadding layer main hum en_voab ka size denge aur embadding ka dim dengi per hr sentance har token ke 100 value banjegi.
_,enc_state=GRU(LATENT,return_state=True)(enc_emb)
### is main hamre pass aisa hota hain keh muje pehla wala output nahe chaye aur muje serf ec
### encoder ka output chaiye final satate muje chaye GRU se hadden satate pass karo aur emb layer bhi.
dec_input=Input(shape=(MAX_UR-1,),name ='gru decoder input ')
dec_emb_obj=Embedding(ur_vocab,EMBADING_DIM)
dec_emb = dec_emb_obj(dec_input)

dec_seq,_= GRU( LATENT, return_sequences=True, return_state=True )(dec_emb,initial_state=enc_state)
gru_outputs = Dense( ur_vocab, activation="softmax" )(dec_seq)
gru_model = Model( [enc_input, dec_input], gru_outputs)


I0000 00:00:1789454548.151502      58 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 12992 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1789454548.153752      58 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13654 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


In [27]:
LATENT = 256
EMBADING_DIM = 100

# =========================
# ENCODER
# =========================

enc_input = Input(
    shape=(MAX_EN,),
    name="encoder_gru_input"
)

enc_emb_obj = Embedding(
    en_vocab,
    EMBADING_DIM
)

enc_emb = enc_emb_obj(enc_input)

encoder_gru = GRU(
    LATENT,
    return_state=True
)

_, enc_state = encoder_gru(enc_emb)


# =========================
# DECODER
# =========================

dec_input = Input(
    shape=(MAX_UR - 1,),
    name="gru_decoder_input"
)

dec_emb_obj = Embedding(
    ur_vocab,
    EMBADING_DIM
)

dec_emb = dec_emb_obj(dec_input)

decoder_gru = GRU(
    LATENT,
    return_sequences=True,
    return_state=True
)

dec_seq, _ = decoder_gru(
    dec_emb,
    initial_state=enc_state
)

decoder_dense = Dense(
    ur_vocab,
    activation="softmax"
)

gru_outputs = decoder_dense(dec_seq)


# =========================
# COMPLETE MODEL
# =========================

gru_model = Model(
    [enc_input, dec_input],
    gru_outputs
)

In [28]:
gru_model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ encoder_gru_input   │ (None, 14)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ gru_decoder_input   │ (None, 9)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_2         │ (None, 14, 100)   │    567,900 │ encoder_gru_inpu… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_3         │ (None, 9, 100)    │    344,200 │ gru_decoder_inpu… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ gru_2 (GRU)         │ [(None, 256),     │    274,944 │ embedding_2[0][0] │
│                     │ (None, 256)]      │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ gru_3 (GRU)         │ [(None, 9, 256),  │    274,944 │ embedding_3[0][0… │
│                     │ (None, 256)]      │            │ gru_2[0][1]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 9, 3442)   │    884,594 │ gru_3[0][0]       │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 2,346,582 (8.95 MB)

 Trainable params: 2,346,582 (8.95 MB)

 Non-trainable params: 0 (0.00 B)

In [34]:
en_seq = en_seq[:8000]

print("en_seq:", en_seq.shape)
print("decoder_input_data:", decoder_input_data.shape)
print("decoder_target:", decoder_target.shape)

en_seq: (8000, 14)
decoder_input_data: (8000, 9)
decoder_target: (8000, 9)


In [35]:
gru_model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [36]:
history = gru_model.fit(
    [en_seq, decoder_input_data],
    decoder_target,
    batch_size=64,
    epochs=20,
    validation_split=0.1
)

Epoch 1/20
113/113 ━━━━━━━━━━━━━━━━━━━━ 7s 19ms/step - accuracy: 0.4260 - loss: 3.9879 - val_accuracy: 0.4839 - val_loss: 3.1737
Epoch 2/20
113/113 ━━━━━━━━━━━━━━━━━━━━ 2s 15ms/step - accuracy: 0.4963 - loss: 3.0514 - val_accuracy: 0.4938 - val_loss: 3.0310
Epoch 3/20
113/113 ━━━━━━━━━━━━━━━━━━━━ 2s 15ms/step - accuracy: 0.5129 - loss: 2.8862 - val_accuracy: 0.5169 - val_loss: 2.9313
Epoch 4/20
113/113 ━━━━━━━━━━━━━━━━━━━━ 2s 15ms/step - accuracy: 0.5261 - loss: 2.7434 - val_accuracy: 0.5288 - val_loss: 2.8374
Epoch 5/20
113/113 ━━━━━━━━━━━━━━━━━━━━ 2s 15ms/step - accuracy: 0.5386 - loss: 2.6029 - val_accuracy: 0.5433 - val_loss: 2.7632
Epoch 6/20
113/113 ━━━━━━━━━━━━━━━━━━━━ 2s 15ms/step - accuracy: 0.5527 - loss: 2.4723 - val_accuracy: 0.5528 - val_loss: 2.7085
Epoch 7/20
113/113 ━━━━━━━━━━━━━━━━━━━━ 2s 15ms/step - accuracy: 0.5629 - loss: 2.3596 - val_accuracy: 0.5603 - val_loss: 2.6759
Epoch 8/20
113/113 ━━━━━━━━━━━━━━━━━━━━ 2s 16ms/step - accuracy: 0.5728 - loss: 2.2592 - val_accu

In [33]:
print("en_seq:", en_seq.shape)
print("decoder_input_data:", decoder_input_data.shape)
print("decoder_target:", decoder_target.shape)

en_seq: (24526, 14)
decoder_input_data: (8000, 9)
decoder_target: (8000, 9)


In [37]:
start_id = ur_tok.word_index["<start>"]
end_id = ur_tok.word_index["<end>"]

print("START ID:", start_id)
print("END ID:", end_id)

step_word = Input(
    shape=(1,),
    name="step_word"
)

step_state_in = Input(
    shape=(LATENT,),
    name="step_state_in"
)

# SAME trained embedding layer
step_emb = dec_emb_obj(step_word)

# SAME trained decoder GRU
step_seq, step_state_out = decoder_gru(
    step_emb,
    initial_state=step_state_in
)

# SAME trained Dense layer
step_probs = decoder_dense(step_seq)

decoder_inference_model = Model(
    [step_word, step_state_in],
    [step_probs, step_state_out]
) 
print("infernce model is ready")

START ID: 1
END ID: 2
infernce model is ready


In [39]:
def translate_sentence(sentence, max_length=MAX_UR):

    # English tokenize
    sequence = en_tok.texts_to_sequences([sentence])

    # Padding
    sequence = pad_sequences(
        sequence,
        maxlen=MAX_EN,
        padding="post"
    )

    # Encoder
    state = encoder_model.predict(
        sequence,
        verbose=0
    )

    # IMPORTANT:
    # Your sequence shows:
    # 1 = START
    # 2 = END
    start_id = 1
    end_id = 2

    current_word = np.array([[start_id]])

    translated_words = []

    # Generate Urdu words
    for _ in range(max_length):

        probabilities, state = decoder_inference_model.predict(
            [current_word, state],
            verbose=0
        )

        predicted_id = np.argmax(
            probabilities[0, 0, :]
        )

        # END token
        if predicted_id == end_id:
            break

        # ID -> Urdu word
        predicted_word = ur_tok.index_word.get(
            predicted_id,
            ""
        )

        if predicted_id != start_id and predicted_word:
            translated_words.append(predicted_word)

        # Next token
        current_word = np.array([[predicted_id]])

    return " ".join(translated_words)